# Read Data

In [83]:
# Cell này giúp xuất DataFrame từ TF-IDF artifacts và giải thích giới hạn khôi phục dữ liệu gốc.
from pathlib import Path
import pickle
import pandas as pd
from scipy import sparse

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "data" / "processed").exists()),
    None,
 )
if project_root is None:
    raise FileNotFoundError("Không tìm thấy thư mục data/processed từ thư mục hiện tại.")

processed_dir = project_root / "data" / "processed"
clean_movies_path = processed_dir / "clean_movies.parquet"
clean_ratings_path = processed_dir / "clean_ratings.parquet"
if not clean_movies_path.exists() and not clean_ratings_path.exists():
    raise FileNotFoundError("Không tìm thấy cả clean_movies.parquet và clean_ratings.parquet")
# Đọc lại df gốc từ clean_movies.parquet (nếu có)
if clean_movies_path.exists():
    movies_df = pd.read_parquet(clean_movies_path)
    print(f"\nĐọc lại df gốc từ {clean_movies_path}: shape={movies_df.shape}")
    print(movies_df.shape)
if clean_ratings_path.exists():
    ratings_df = pd.read_parquet(clean_ratings_path)
    print(f"\nĐọc lại ratings df từ {clean_ratings_path}: shape={ratings_df.shape}")


collab_npz_path = processed_dir / "collab_matrix_normalized.npz"
collab_pkl_path = processed_dir / "collab_mappings.pkl"
collab_pkl_data = None
if collab_npz_path.exists():
    collab_mat = sparse.load_npz(collab_npz_path)
if collab_pkl_path.exists():
    print(f"\nĐọc collaborative matrix từ {collab_pkl_path}")
    with open(collab_pkl_path, "rb") as f:
        collab_pkl_data = pickle.load(f)
if isinstance(collab_pkl_data, (list, tuple)) or (hasattr(collab_pkl_data, 'shape') and len(collab_pkl_data.shape) == 1):
        print(f"Số lượng phần tử: {len(collab_pkl_data)}")
        print(f"5 phần tử đầu tiên: {collab_pkl_data[:5]}")
        # Gán vào biến để sử dụng
        movie_ids = collab_pkl_data

tfidf_npz_path = processed_dir / "tfidf_matrix.npz"
tfidf_pkl_path = processed_dir / "tfidf_matrix.pkl"
tfidf_pkl_data = None
if tfidf_npz_path.exists():
    tfidf_mat = sparse.load_npz(tfidf_npz_path)
if tfidf_pkl_path.exists():
    print(f"\nĐọc TF-IDF matrix từ {tfidf_pkl_path}")
    with open(tfidf_pkl_path, "rb") as f:
        tfidf_pkl_data = pickle.load(f)
if isinstance(tfidf_pkl_data, (list, tuple)) or (hasattr(tfidf_pkl_data, 'shape') and len(tfidf_pkl_data.shape) == 1):
        print(f"Số lượng phần tử: {len(tfidf_pkl_data)}")
        print(f"5 phần tử đầu tiên: {tfidf_pkl_data[:5]}")
        # Gán vào biến để sử dụng
        tfidf_matrix = tfidf_pkl_data    






Đọc lại df gốc từ e:\Year_3\Semester_2\Machine Learning\BTL\video_rec_system\data\processed\clean_movies.parquet: shape=(46625, 6)
(46625, 6)

Đọc lại ratings df từ e:\Year_3\Semester_2\Machine Learning\BTL\video_rec_system\data\processed\clean_ratings.parquet: shape=(11412350, 3)

Đọc collaborative matrix từ e:\Year_3\Semester_2\Machine Learning\BTL\video_rec_system\data\processed\collab_mappings.pkl

Đọc TF-IDF matrix từ e:\Year_3\Semester_2\Machine Learning\BTL\video_rec_system\data\processed\tfidf_matrix.pkl


In [84]:
print(movies_df.id.nunique())  # In ra số lượng phim duy nhất
print(ratings_df.userId.nunique()) # In ra số lượng người dùng duy nhất
print(ratings_df.movieId.nunique()) # In ra số lượng phim được đánh giá duy nhất

print(ratings_df.head())  # In ra một phần nhỏ của ratings_df để kiểm tra

mean_ratings_per_user = ratings_df.groupby('userId').size().mean()
print(f"\nSố lượng đánh giá trung bình mỗi người dùng: {mean_ratings_per_user:.2f}")

list_of_user_has_more_than_ = ratings_df.groupby('userId').size().tolist()

print(collab_mat.shape)  # In ra shape của collaborative matrix
print(tfidf_mat.shape)  # In ra shape của TF-IDF matrix

print(collab_mat.data[:5][:5])  # In ra một phần nhỏ của ma trận để kiểm tra
print(tfidf_mat.data[:5][:5])  # In ra một phần nhỏ của ma trận để kiểm tra

45430
255451
5346
   userId  movieId  rating
0       1      110     1.0
1       1      147     4.5
2       1      858     5.0
4       1     1246     5.0
5       1     1968     4.0

Số lượng đánh giá trung bình mỗi người dùng: 44.68
(255451, 5346)
(46625, 10000)
[-3.4210513   0.33962005  0.33228922  0.6304486  -0.28142002]
[0.68059415 0.12094443 0.09723533 0.0526435  0.08350578]


# Movie_Feature

In [85]:
import numpy as np
import scipy.sparse as sparse
from sklearn.decomposition import TruncatedSVD

class MovieFeatureExtractor:
    """
    Lớp trích xuất đặc trưng nội dung phim (Content-Based Filtering)
    sử dụng Latent Semantic Analysis (LSA) trên ma trận thưa TF-IDF.
    """
    def __init__(self, k_dimensions=100, random_state=42):
        """
        Khởi tạo Extractor.
        
        Parameters:
        - k_dimensions: Số lượng chủ đề ẩn (Latent topics) muốn giữ lại.
        - random_state: Seed để tái tạo lại kết quả.
        """
        self.k_dimensions = k_dimensions
        self.random_state = random_state
        self.variance_retained = 0.0
        
        # Khởi tạo mô hình TruncatedSVD
        self.svd_model = TruncatedSVD(
            n_components=self.k_dimensions, 
            algorithm='randomized', 
            random_state=self.random_state
        )

    def fit(self, tfidf_mat):
        """
        Huấn luyện mô hình để tìm ra không gian vector ẩn (Latent space).
        
        Parameters:
        - tfidf_mat: Ma trận scipy.sparse cần để train.
        
        Returns:
        - self: Trả về chính object hiện tại.
        """
        print(f"[*] Đang huấn luyện mô hình SVD (Fit) với k = {self.k_dimensions}...")
        
        # Fit ma trận TF-IDF để học U, Sigma, V^T
        self.svd_model.fit(tfidf_mat)
        
        # Phân tích phương sai (Độ tin cậy của mô hình)
        self.variance_retained = np.sum(self.svd_model.explained_variance_ratio_) * 100
        
        print(f"[+] Huấn luyện hoàn tất!")
        print(f"[+] Tổng lượng thông tin được giữ lại (Explained Variance): {self.variance_retained:.2f}%")
        
        # Cảnh báo học thuật
        if self.variance_retained < 60:
            print("[!] Cảnh báo: Lượng thông tin giữ lại < 60%. Bạn nên cân nhắc tăng k_dimensions.")
            
        return self

    def transform(self, tfidf_mat):
        """
        Chiếu ma trận TF-IDF vào không gian ẩn đã được học từ bước fit.
        
        Parameters:
        - tfidf_mat: Ma trận scipy.sparse cần biến đổi.
        
        Returns:
        - item_content_profiles: Ma trận đặc trưng nội dung của video (Dense array)
        """
        print(f"[*] Đang biến đổi dữ liệu (Transform)...")
        
        # Transform dữ liệu
        item_content_profiles = self.svd_model.transform(tfidf_mat)
        
        print(f"[+] Kích thước Item Content Profiles: {item_content_profiles.shape}")
        return item_content_profiles

    def fit_transform(self, tfidf_mat):
        """
        Kết hợp fit và transform trong 1 bước (tối ưu hóa hiệu năng tính toán).
        """
        self.fit(tfidf_mat)
        return self.transform(tfidf_mat)


# ==========================================
# --- THỰC THI CHƯƠNG TRÌNH ---
# ==========================================

# Giả lập tạo một ma trận thưa TF-IDF ngẫu nhiên (46625 movies x 10000 keywords)
# Trong thực tế, bạn sẽ dùng: tfidf_mat = sparse.load_npz(npz_path)
print("[-] Đang tạo ma trận TF-IDF giả lập...")
tfidf_mat = tfidf_mat

# 1. Khởi tạo class
extractor = MovieFeatureExtractor(k_dimensions=100)

# 2. Fit và Transform
# Cách 1: Chạy fit riêng và transform riêng (Hữu ích khi train xong muốn lưu model lại)


# Cách 2: Chạy fit_transform cùng lúc cho tập train
movie_factors_matrix = extractor.fit_transform(tfidf_mat)
print(f"3. Kích thước ma trận trọng số đã học: {extractor.svd_model.components_.shape}")

# 3. Kiểm tra kết quả
non_zero_count = np.count_nonzero(movie_factors_matrix)
total_elements = movie_factors_matrix.size

print("-" * 40)
print(f"[+] Số lượng phần tử khác 0: {non_zero_count}") 
print(f"[+] Tỷ lệ phần tử khác 0 (Sparsity check): {(non_zero_count / total_elements) * 100:.4f}%")

# Lưu trữ artifact để phục vụ việc huấn luyện Meta-learner
# np.save('item_content_factors_k100.npy', item_cbf_factors)

[-] Đang tạo ma trận TF-IDF giả lập...
[*] Đang huấn luyện mô hình SVD (Fit) với k = 100...
[+] Huấn luyện hoàn tất!
[+] Tổng lượng thông tin được giữ lại (Explained Variance): 11.88%
[!] Cảnh báo: Lượng thông tin giữ lại < 60%. Bạn nên cân nhắc tăng k_dimensions.
[*] Đang biến đổi dữ liệu (Transform)...
[+] Kích thước Item Content Profiles: (46625, 100)
3. Kích thước ma trận trọng số đã học: (100, 10000)
----------------------------------------
[+] Số lượng phần tử khác 0: 4653800
[+] Tỷ lệ phần tử khác 0 (Sparsity check): 99.8134%


# Creating Data

In [86]:
import numpy as np
userid_to_index_collab = collab_pkl_data['user_mapping']
index_to_userid_collab = {v: k for k, v in userid_to_index_collab.items()}

movieid_to_index_collab = collab_pkl_data['movie_mapping']
index_to_movieid_collab = {v: k for k, v in movieid_to_index_collab.items()}

movieid_to_index_tfidf = tfidf_pkl_data['movie_mapping']
index_to_movieid_tfidf = {v: k for k, v in movieid_to_index_tfidf.items()}


rows, cols = collab_mat.nonzero()
values = collab_mat.data
print(type(movie_factors_matrix))
print(movie_factors_matrix.shape)  # In ra shape của ma trận trọng số đã học

X_list = []
y_list = []
for user_idx_in_collab, movie_idx_in_collab, rating in zip(rows, cols, values):
    user_id = index_to_userid_collab.get(user_idx_in_collab, "Unknown User")
    movie_id = index_to_movieid_collab.get(movie_idx_in_collab, "Unknown Movie")
    movie_idx_in_tfidf = movieid_to_index_tfidf.get(movie_id, None)

    if movie_idx_in_tfidf is None:
        continue  # bỏ nếu không map được

    row = movie_factors_matrix[movie_idx_in_tfidf]  # (100,) hoặc (1,100)

    # đảm bảo shape đúng (1D vector)
    row = row.flatten()

    X_list.append(row)
    y_list.append(rating)

X = np.array(X_list)  # shape: (n_samples, 100)
y = np.array(y_list)  # shape: (n_samples,)

print(X.shape)
print(y.shape)
print(X[0])
print(y[0])
    

<class 'numpy.ndarray'>
(46625, 100)
(11412350, 100)
(11412350,)
[ 9.13597494e-02 -8.49442836e-03 -5.11860475e-02 -1.39168305e-02
 -3.36016044e-02  1.40808085e-02  2.74146926e-02 -1.14594623e-02
 -9.05508026e-02 -1.77534446e-02  1.68550178e-01  1.68077856e-01
  3.14605162e-02  5.34104183e-03  5.97767010e-02 -6.24463614e-03
 -6.81631267e-02 -1.61131267e-02 -9.91148967e-03  5.28466739e-02
  3.24083231e-02 -3.04645207e-02 -2.71109808e-02 -4.11366858e-02
 -9.64039564e-03 -2.44965460e-02  1.68936811e-02 -1.52578736e-02
 -4.07865569e-02 -1.26716541e-02 -8.58681742e-03 -1.54835377e-02
  1.77137437e-03 -4.52238275e-03 -2.36646030e-02  2.96341702e-02
 -1.08373417e-02 -2.31142733e-02  1.73873908e-03 -2.07578577e-02
  3.78574319e-02  1.10624125e-04 -1.42249195e-02 -1.29236504e-02
 -2.75071878e-02  1.45808524e-02 -2.15979293e-02 -8.68715439e-03
 -1.41135240e-02  9.93252615e-04 -1.39745316e-02  5.30254515e-03
  1.12771261e-02 -4.99190623e-03 -2.31819134e-02 -8.17192812e-03
  3.05341799e-02  4.20557

# Traning

In [87]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ---------------------------------------------------------
# 0. Chuẩn bị dữ liệu (Giả sử bạn đã có X và y)
# LỜI KHUYÊN: Ép kiểu X về float32 để giảm một nửa dung lượng RAM 
# (từ ~9GB xuống còn ~4.5GB)
# ---------------------------------------------------------
# X = np.array(X_list, dtype=np.float32) 
# y = np.array(y_list, dtype=np.float32)

print("1. Đang chia tập dữ liệu...")
# Chia 80% để Train, 20% để Validation
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("2. Đang tạo LightGBM Dataset...")
# Khởi tạo Dataset của LightGBM. 
# free_raw_data=True giúp giải phóng RAM từ mảng NumPy ban đầu sau khi tạo Dataset
train_data = lgb.Dataset(X_train, label=y_train, free_raw_data=True)
valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data, free_raw_data=True)

print("3. Thiết lập thông số (Hyperparameters)...")
params = {
    'objective': 'regression',       # Bài toán dự đoán số thực (điểm rating)
    'metric': 'rmse',                # Đo lường sai số bằng Root Mean Squared Error
    'boosting_type': 'gbdt',         # Gradient Boosting Decision Tree truyền thống
    'learning_rate': 0.01,           # Tốc độ học (nhỏ thì mô hình học kỹ hơn nhưng lâu hơn)
    'num_leaves': 63,                # Số lá tối đa trên mỗi cây (Tăng lên 63 vì data bạn rất lớn)
    'max_depth': -1,                 # Không giới hạn độ sâu (LightGBM tự ưu tiên phát triển theo chiều sâu của lá)
    'feature_fraction': 0.8,         # Mỗi cây chỉ lấy ngẫu nhiên 80% trong số 100 features để chống overfit
    'bagging_fraction': 0.8,         # Lấy ngẫu nhiên 80% data cho mỗi vòng lặp
    'bagging_freq': 5,               # Tần suất thực hiện bagging
    'n_jobs': -1,                    # Tận dụng TẤT CẢ các luồng CPU đang có
    # 'device': 'gpu'                # BỎ COMMENT dòng này nếu máy bạn có GPU đã cài đặt CUDA
}

print("4. Bắt đầu huấn luyện...")
# Sử dụng API mới của LightGBM với callbacks
model = lgb.train(
    params,
    train_data,
    num_boost_round=5000,            # Cho phép tạo tối đa 5000 cây
    valid_sets=[train_data, valid_data],
    valid_names=['train', 'valid'],
    callbacks=[
        # Dừng quá trình train nếu sau 50 cây mà điểm trên tập Validation không giảm
        lgb.early_stopping(stopping_rounds=50), 
        # In kết quả ra màn hình sau mỗi 100 cây
        lgb.log_evaluation(period=100)          
    ]
)

print("5. Đánh giá mô hình trên tập Validation...")
# Dự đoán trên tập Validation
y_pred = model.predict(X_valid)

# Tính toán sai số
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
mae = mean_absolute_error(y_valid, y_pred)

print(f"\n--- KẾT QUẢ ---")
print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"MAE (Mean Absolute Error): {mae:.4f}")

# (Tùy chọn) Xem mức độ quan trọng của các Features
# Cho biết trong 100 features của bạn, thông tin nào đóng góp nhiều nhất vào việc đoán điểm
# import matplotlib.pyplot as plt
# lgb.plot_importance(model, max_num_features=20, figsize=(10, 6))
# plt.show()

1. Đang chia tập dữ liệu...
2. Đang tạo LightGBM Dataset...
3. Thiết lập thông số (Hyperparameters)...
4. Bắt đầu huấn luyện...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.254612 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25494
[LightGBM] [Info] Number of data points in the train set: 9129880, number of used features: 100
[LightGBM] [Info] Start training from score 0.000218
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	train's rmse: 0.872782	valid's rmse: 0.872965
5. Đánh giá mô hình trên tập Validation...

--- KẾT QUẢ ---
RMSE (Root Mean Squared Error): 0.8730
MAE (Mean Absolute Error): 0.6712
